In [ ]:
#import libraries
import pandas as pd

**Task 4**: Financial Risk Identification


● Track accounts with frequent large withdrawals or overdrafts

In [ ]:
df=pd.read_csv("/content/3rd dataset.morgan.csv")

In [ ]:
#Find Large Withdrawals
large_withdrawals = df[
    (df["TransactionType"] == "Withdrawal") &
    (df["TransactionAmount"] >= 75000)
]
print(large_withdrawals)

     TransactionID CustomerID AccountID AccountType TransactionType  \
3              180   CUST2541  ACC10117     Current      Withdrawal   
6               66   CUST6028  ACC10117        Loan      Withdrawal   
12             134   CUST6082  ACC10996      Credit      Withdrawal   
27              34   CUST4794  ACC11285     Savings      Withdrawal   
31             173   CUST4780  ACC11837     Savings      Withdrawal   
52              90   CUST8155  ACC15671        Loan      Withdrawal   
54             143   CUST2942  ACC15671     Current      Withdrawal   
61              29   CUST3041  ACC16664      Credit      Withdrawal   
95              90   CUST2349  ACC19178      Credit      Withdrawal   
105            185   CUST1497  ACC21264      Credit      Withdrawal   
140             31   CUST6947  ACC24070     Savings      Withdrawal   
142            185   CUST4780  ACC24508      Credit      Withdrawal   
177            167   CUST3069  ACC28292     Current      Withdrawal   
179   

In [ ]:
#Count Large Withdrawals for Each Account
withdrawal_count = large_withdrawals.groupby("AccountID").size().reset_index(name="LargeWithdrawalCount")

print(withdrawal_count)

   AccountID  LargeWithdrawalCount
0   ACC10117                     2
1   ACC10996                     1
2   ACC11285                     1
3   ACC11837                     1
4   ACC15671                     2
5   ACC16664                     1
6   ACC19178                     1
7   ACC21264                     1
8   ACC24070                     1
9   ACC24508                     1
10  ACC28292                     2
11  ACC29231                     2
12  ACC29396                     1
13  ACC32627                     1
14  ACC33287                     2
15  ACC34119                     1
16  ACC34431                     1
17  ACC34568                     1
18  ACC34821                     1
19  ACC35419                     1
20  ACC37688                     2
21  ACC41829                     1
22  ACC43771                     1
23  ACC45968                     1
24  ACC48303                     1
25  ACC49774                     1
26  ACC50817                     1
27  ACC53466        

In [ ]:
#Flag Frequent Large Withdrawals
frequent_large_withdrawals = withdrawal_count[
    withdrawal_count["LargeWithdrawalCount"] >= 2
]

print(frequent_large_withdrawals)

   AccountID  LargeWithdrawalCount
0   ACC10117                     2
4   ACC15671                     2
10  ACC28292                     2
11  ACC29231                     2
14  ACC33287                     2
20  ACC37688                     2
36  ACC77592                     2


In [ ]:
#Overdraft Accounts
overdraft_accounts = df[df["AccountBalance"] < 0]

overdraft_accounts = overdraft_accounts.groupby("AccountID").size().reset_index(name="OverdraftCount")

print(overdraft_accounts)

   AccountID  OverdraftCount
0   ACC11837               1
1   ACC17688               1
2   ACC18177               1
3   ACC19156               1
4   ACC26940               1
5   ACC32212               1
6   ACC34431               1
7   ACC39529               1
8   ACC48501               1
9   ACC49774               1
10  ACC50817               1
11  ACC53466               1
12  ACC58667               1
13  ACC60432               1
14  ACC70741               1
15  ACC73104               1
16  ACC75675               1
17  ACC77773               1
18  ACC78589               1
19  ACC89098               1
20  ACC97411               1


**4.2** Calculate balance volatility using standard deviation or coefficient of variation.


In [ ]:
# standard deviation
balance_volatility = df.groupby("AccountID")["AccountBalance"].std().reset_index()

balance_volatility.rename(columns={"AccountBalance": "BalanceStdDev"}, inplace=True)

print(balance_volatility.head())

  AccountID  BalanceStdDev
0  ACC10117   38036.539989
1  ACC10996   30036.385595
2  ACC11062   33831.704345
3  ACC11188    9067.866694
4  ACC11285   25590.622179


In [ ]:
#coefficient of variation.
balance_volatility = df.groupby("AccountID")["AccountBalance"].agg(
    MeanBalance="mean",
    StdDev="std"
).reset_index()

balance_volatility["CoefficientOfVariation"] = (
    balance_volatility["StdDev"] /
    balance_volatility["MeanBalance"]
)

print(balance_volatility.head())

  AccountID   MeanBalance        StdDev  CoefficientOfVariation
0  ACC10117  94082.590727  38036.539989                0.404289
1  ACC10996  72517.761136  30036.385595                0.414194
2  ACC11062  73541.328302  33831.704345                0.460037
3  ACC11188  51359.039950   9067.866694                0.176558
4  ACC11285  79923.619870  25590.622179                0.320188


**4.3** Use IQR or z-score methods to detect anomalies.


In [ ]:
#Calculate Quartiles
Q1 = df["TransactionAmount"].quantile(0.25)
Q3 = df["TransactionAmount"].quantile(0.75)

IQR = Q3 - Q1

In [ ]:
#Calculate Lower and Upper Limits
lower_limit = Q1 - (1.5 * IQR)
upper_limit = Q3 + (1.5 * IQR)

In [ ]:
#Find Anomalies
anomalies = df[
    (df["TransactionAmount"] < lower_limit) |
    (df["TransactionAmount"] > upper_limit)
]

print(anomalies)

     TransactionID CustomerID AccountID AccountType TransactionType  \
98              13   CUST2805  ACC19178      Credit         Payment   
135             47   CUST7098  ACC23736        Loan         Payment   
273             52   CUST4258  ACC35419        Loan        Transfer   
623             92   CUST6526  ACC77533        Loan        Transfer   
745            131   CUST4769  ACC92104     Current         Deposit   

           Product    Firm   Region    Manager TransactionDate  ...  \
98     Credit Card  Firm D     West  Manager 3      2024-02-17  ...   
135      Home Loan  Firm E     West  Manager 3      2023-09-28  ...   
273  Personal Loan  Firm B  Central  Manager 1      2023-11-07  ...   
623    Mutual Fund  Firm D     West  Manager 3      2024-02-17  ...   
745    Mutual Fund  Firm B     East  Manager 1      2024-01-17  ...   

     TenureMonths  Year      Month     NetAmount  PreviousTransaction  \
98            126  2024   February -138133.82160           2023-11-06   


In [ ]:
#Count Anomalies
print("Number of Anomalies:", len(anomalies))

Number of Anomalies: 5


In [ ]:
#View Important Columns
print(anomalies[[
    "AccountID",
    "TransactionDate",
    "TransactionType",
    "TransactionAmount"
]])

    AccountID TransactionDate TransactionType  TransactionAmount
98   ACC19178      2024-02-17         Payment       138133.82160
135  ACC23736      2023-09-28         Payment       141210.80340
273  ACC35419      2023-11-07        Transfer       -36173.69555
623  ACC77533      2024-02-17        Transfer       154662.04580
745  ACC92104      2024-01-17         Deposit       144137.88030


In [ ]:
#Method 2: Z-Score
from scipy.stats import zscore

df["ZScore"] = zscore(df["TransactionAmount"])

In [ ]:
anomalies = df[
    (df["ZScore"] > 3) |
    (df["ZScore"] < -3)
]

print(anomalies)

     TransactionID CustomerID AccountID AccountType TransactionType  \
273             52   CUST4258  ACC35419        Loan        Transfer   
623             92   CUST6526  ACC77533        Loan        Transfer   
745            131   CUST4769  ACC92104     Current         Deposit   

           Product    Firm   Region    Manager TransactionDate  ...  Year  \
273  Personal Loan  Firm B  Central  Manager 1      2023-11-07  ...  2023   
623    Mutual Fund  Firm D     West  Manager 3      2024-02-17  ...  2024   
745    Mutual Fund  Firm B     East  Manager 1      2024-01-17  ...  2024   

        Month     NetAmount  PreviousTransaction  GapDays   Status  \
273  November   36173.69555           2023-07-06    124.0  Dormant   
623  February -154662.04580           2023-06-18    244.0  Dormant   
745   January  144137.88030           2023-11-19     59.0   Active   

    AverageBalance  TransactionVolume CustomerSegment    ZScore  
273   62737.431942                  4           Basic -3.15

**4.4** Highlight customers with irregular or suspicious transaction behavior.


In [ ]:
#Create a Suspicious Status column
df["SuspiciousStatus"] = "Normal"

In [ ]:
#Flag customers with negative balance
df.loc[df["AccountBalance"] < 0, "SuspiciousStatus"] = "Suspicious"

In [ ]:
#Flag customers with anomalous transactions
df.loc[
    (df["TransactionAmount"] < lower_limit) |
    (df["TransactionAmount"] > upper_limit),
    "SuspiciousStatus"
] = "Suspicious"

In [ ]:
#Flag customers with frequent large withdrawals
suspicious_accounts = frequent_large_withdrawals["AccountID"]

df.loc[
    df["AccountID"].isin(suspicious_accounts),
    "SuspiciousStatus"
] = "Suspicious"

In [ ]:
#View suspicious customers
suspicious_customers = df[df["SuspiciousStatus"] == "Suspicious"]

print(suspicious_customers[
    ["AccountID",
     "TransactionDate",
     "TransactionType",
     "TransactionAmount",
     "AccountBalance"]
])

    AccountID TransactionDate TransactionType  TransactionAmount  \
0    ACC10117      2023-01-30         Deposit        52336.34278   
1    ACC10117      2023-04-30        Transfer        77018.97098   
2    ACC10117      2023-06-20        Transfer        65805.20213   
3    ACC10117      2023-07-14      Withdrawal        89395.76266   
4    ACC10117      2023-08-04      Withdrawal       -18353.15647   
..        ...             ...             ...                ...   
639  ACC77773      2023-10-06         Deposit        62854.65269   
655  ACC78589      2023-10-06         Payment        46752.23974   
728  ACC89098      2023-09-10      Withdrawal       108392.92230   
745  ACC92104      2024-01-17         Deposit       144137.88030   
785  ACC97411      2023-12-24         Payment       123364.04070   

     AccountBalance  
0      46657.217900  
1     180539.287700  
2      80204.673020  
3      70489.109440  
4      99972.730020  
..              ...  
639    -2427.573420  
655   -

In [ ]:
#Count suspicious customers
print(df["SuspiciousStatus"].value_counts())

SuspiciousStatus
Normal        731
Suspicious     69
Name: count, dtype: int64


In [ ]:
df.to_csv("4th dataset morgan.csv",index=False)